# NLP Basics Assessment  -

INTEGRANTES: ÁLVARO JOSÉ CABRERA LOZANO , CLAUDIA LORENA ARAGÓN, JOSUE COBALEDA

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ohtar10/icesi-nlp/blob/main/Sesion1/6-practice.ipynb)

En este notebook vamos a poner en práctica algunos de los conceptos vistos en los notebooks anteriores, aplicado a un corpus específico:
[_An Occurrence at Owl Creek Bridge_](https://en.wikipedia.org/wiki/An_Occurrence_at_Owl_Creek_Bridge) por Ambrose Bierce (1890). Esta historia es de dominio público y el corpus fue obtenido de [Project Gutenberg](https://www.gutenberg.org/ebooks/375.txt.utf-8).

## Referencias
* [NLP - Natural Language Processing With Python](https://www.udemy.com/course/nlp-natural-language-processing-with-python)
* [Natural Language Processing in Action](https://www.manning.com/books/natural-language-processing-in-action)

In [1]:
import pkg_resources
import warnings

warnings.filterwarnings('ignore')

installed_packages = [package.key for package in pkg_resources.working_set]
IN_COLAB = 'google-colab' in installed_packages

/tmp/ipython-input-2396000874.py:1: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  import pkg_resources


In [2]:
!test '{IN_COLAB}' = 'True' && wget https://raw.githubusercontent.com/alvarojcabrera/icesi-nlp/main/requirements.txt && pip install -r requirements.txt

--2026-02-21 03:47:42--  https://raw.githubusercontent.com/alvarojcabrera/icesi-nlp/main/requirements.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.111.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 349 [text/plain]
Saving to: ‘requirements.txt.1’

requirements.txt.1  100%[===================>]     349  --.-KB/s    in 0s      

2026-02-21 03:47:42 (12.6 MB/s) - ‘requirements.txt.1’ saved [349/349]



In [3]:
# RUN THIS CELL to perform standard imports:
import spacy
nlp = spacy.load('en_core_web_sm')

# Ahora, el objetivo es utilizar un Libro, en este caso uno de mis favoritos: Juego de Tronos, por lo que voy a utilizar una libreria capaz de convertir EPUB (tipico de los libros electronicos) en .txt

# Para eso usaremos epub2txt

In [4]:
!pip install epub2txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.6/111.6 kB 4.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.8/127.8 kB 12.2 MB/s eta 0:00:00
  Created wheel for ebooklib: filename=EbookLib-0.17.1-py3-none-any.whl size=38170 sha256=176d19ac1e00e8b7571beed21d98c84b20c92bf77acd7f9e4a93d321630f8a40
  Stored in directory: /root/.cache/pip/wheels/26/70/17/1dce3a7603cf4e38cf35bd928607bdf26982bec402d612e19a
Successfully built ebooklib
  Attempting uninstall: absl-py
    Found existing installation: absl-py 1.4.0
    Uninstalling absl-py-1.4.0:
      Successfully uninstalled absl-py-1.4.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.19.0 requires absl-py>=1.0.0, but you have absl-py 0.11.0 which is incompatible.
tensorflow 2.19.0 requires tensorboard~=2.19.0, but you have tensorboard 2.16.2 wh

In [6]:
import epub2txt
from pathlib import Path

epub_path = "/content/A Game of Thrones -- George R_ R_ Martin -- A Song of Ice and Fire, Book One, Bantam Spectra Trade -- Bantam Books -- 9780553381689 -- b992e8ac9d53df567b785e5c049839cf -- Anna’s Archive.epub"      # tu archivo epub
txt_path  = "got.txt"

text = epub2txt.epub2txt(epub_path)

# (Opcional) limpieza ligera: quita líneas vacías repetidas
lines = [ln.rstrip() for ln in text.splitlines()]
clean = "\n".join(lines)

Path(txt_path).write_text(clean, encoding="utf-8")
print("Listo ✅", txt_path)

Listo ✅ got.txt


Sin embargo... un libro es un documento muy largo, con encabezados, mapas, indices y este tipo de cosas. Para este ejercicio, luchamos mucho con las maneras más correctas de "limpiar" el documento. Asi que decidimos...

que aveces lo más simple es lo mejor, pues teniendo ahora el documento en txt, retiramos manual todo lo innecesario y dejamos solamente el prologo y el capitulo 1 para poder probar un poco el ejercicio! esperamos a lo largo del curso aprender a limpiar bien estos asuntos para volver a este ejercicio y hacerlo con lo aprendido!

In [13]:
!test '{IN_COLAB}' = 'True' && wget https://raw.githubusercontent.com/alvarojcabrera/icesi-nlp/main/Sesion1/got_clean.txt

--2026-02-21 03:55:28--  https://raw.githubusercontent.com/alvarojcabrera/icesi-nlp/main/Sesion1/got_clean.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 37744 (37K) [text/plain]
Saving to: ‘got_clean.txt’

got_clean.txt       100%[===================>]  36.86K  --.-KB/s    in 0.004s  

2026-02-21 03:55:28 (8.39 MB/s) - ‘got_clean.txt’ saved [37744/37744]



In [14]:
with open('./got_clean.txt') as file:
    doc = nlp(file.read())

En la parte de arriba, en nuestros anteriores intentos, nos limitaba a 1,000,000 de caracteres y debiamos poner un limitante. Intentamos con 500,000 y funcionó, sin embargo ahora con la limpieza "manual" no es necesario establecer limitantes, pues solo es prologo y cap 1

In [16]:
doc[:35]


Prologue
We should start back,” Gared urged as the woods began to grow dark around them. “The wildlings are dead.”“Do the dead frighten you?” Ser

El documento fue cargado exitosamente!

**2. Cuantos tokens hay en el archivo?**

In [17]:
len(doc)

8372

**3. Cuantas oraciones hay en el archivo?**
<br>Pista: Necesitarás una lista primero

In [18]:
sentences = list(doc.sents)
len(sentences)

657

**4. Imprime la segunda oración del documento**
<br> Pista: Los índices comienzan en 0 y el título cuenta como la primera oración.

In [19]:
sentences[1]

“The wildlings are dead.

**5. Por cada token en la oración anterior, imprime su `text`, `POS` tag, `dep` tag y `lemma`**
<br>

In [22]:
print("{:20}{:20}{:20}{:20}".format("Text", "POS", "dep", "lemma"))
for token in sentences[1]:
    print(f"{token.text:{20}}{token.pos_:{20}}{token.dep_:{20}}{token.lemma_:{20}}")

Text                POS                 dep                 lemma               
“                   PUNCT               punct               "                   
The                 DET                 det                 the                 
wildlings           NOUN                nsubj               wildling            
are                 AUX                 ROOT                be                  
dead                ADJ                 acomp               dead                
.                   PUNCT               punct               .                   


Ahora implementemos un matcher que nos muestre todas las menciones de "direwolf" o "direwolves" que son el Lobo Huargo! Blasón insignia de la casa Stark

In [23]:
from spacy.matcher import Matcher

matcher = Matcher(nlp.vocab)

pattern = [{"LOWER": {"IN": ["direwolf", "direwolves"]}}]
matcher.add("Direwolf", [pattern])

matches = matcher(doc)
len(matches)


8

In [25]:
for _, start, end in matches[:5]:
    print(doc[start-6:end+6].text)
    print("----")



Gared said. “Bears and direwolves and . . . and other
----
Starks of Winterfell: a grey direwolf racing across an ice-white
----
calmly. “That’s a direwolf. They grow larger than the
----
“There’s not been a direwolf sighted south of the Wall in
----
sons, two daughters. The direwolf is the sigil of your House
----


In [26]:
for sentence in sentences:
    for _, start, end in matches:
        if sentence.start <= start and sentence.end >= end:
            print(sentence.text, "\n")

“Bears and direwolves and . . . 

Over their heads flapped the banner of the Starks of Winterfell: a grey direwolf racing across an ice-white field. 

“That’s a direwolf. 

Greyjoy said, “There’s not been a direwolf sighted south of the Wall in two hundred years. 

The direwolf is the sigil of your House. 

“The direwolf graces the banners of House Stark,” Jon pointed out. 

A direwolf will rip a man’s arm off his shoulder as easily as a dog will kill a rat. 

They watched him dismount where the direwolf lay dead in the snow, watched him kneel. 



Jugemos un poco!

Que personaje aparecen más? En este pequeño prologo y capitulo1, claro

In [27]:
from collections import Counter

proper_nouns = [token.text for token in doc if token.pos_ == "PROPN"]
Counter(proper_nouns).most_common(10)

[('Bran', 37),
 ('Jon', 29),
 ('Waymar', 24),
 ('Robb', 24),
 ('Ser', 22),
 ('Royce', 22),
 ('lord', 15),
 ('Greyjoy', 13),
 ('Jory', 11),
 ('father', 10)]

El capitulo 1 del libro pertenece a Bran, por lo que tiene sentido que sea el mas nombrado.

Por otro lado, el prologo pertenece a Royce, un Caballero o "Ser". Por lo que su titulo SIEMPRE debe ir antes que su nombre, cosa que concuerda, pues tenemos 22 apariciones de Royce y 22 de Ser, pues de momento ha sido el unico caballero que ha aparecido en el prologo y en el capitulo 1.

Ahora veamos cuantos adjetivos hay...

In [28]:
sum(1 for token in doc if token.pos_ == "ADJ")

479

Y verbos?

In [29]:
sum(1 for token in doc if token.pos_ == "VERB")

1062

El norte es muy frio, cuantas veces aparece la palabra "cold"?

In [32]:
cold_sentences = [
    s for s in doc.sents
    if any(t.lemma_.lower() == "cold" for t in s)
]

print(f"Hay {len(cold_sentences)} oraciones relacionadas con 'cold'.")

Hay 24 oraciones relacionadas con 'cold'.


In [33]:
for sentence in doc.sents:
    if "cold" in sentence.text.lower():
        print(sentence.text)

A cold wind was blowing out of the north, and it made the trees rustle like living things.
All day, Will had felt as though something were watching him, something cold and implacable that loved him not.
“It was the cold,” Gared said with iron certainty.
Everyone talks about snows forty foot deep, and how the ice wind comes howling out of the north, but the real enemy is the cold.
Nothing burns like the cold.
”“I’ve had the cold in me too, lordling.”
Gared said it was the cold . . . ”
It wasn’t cold enough.”Royce nodded.
We’ve had a few light frosts this past week, and a quick flurry of snow now and then, but surely no cold fierce enough to kill eight grown men.
A cold wind whispered through the trees.
The taste of cold iron in his mouth gave him comfort.
Why is it so cold?”It was cold.
It was very cold.
His hands trembled from the weight of it, or perhaps from the cold.
They fixed on the longsword trembling on high, watched the moonlight running cold along the metal.
Ser Waymar may hav

Busquemos ahora patrones... Veamos cuando un personaje interactua...


In [34]:
pattern = [
    {"POS": "PROPN"},
    {"LOWER": {"IN": ["said","asked","replied","urged"]}}
]

matcher = Matcher(nlp.vocab)
matcher.add("SpeechVerb", [pattern])

matches = matcher(doc)

for _, start, end in matches[:10]:
    print(doc[start:end].text)

Royce asked
Royce asked
Royce replied
“Mormont said
Royce asked
Waymar asked
Gared said
Royce said
Waymar asked
lord said


¿Cuantas oraciones contienen dialogo? En inglés los dialogos se suelen expresar en comillas... entonces veamos.

In [35]:
dialogue_sentences = [s for s in sentences if "“" in s.text or '"' in s.text]
len(dialogue_sentences)

175

Y en promedio cuanto dura una oración?

In [36]:
import numpy as np

lengths = [len(sentence) for sentence in sentences]
np.mean(lengths)

12.742770167427702